# Flow Configuration Testing

This notebook tests the FlowConfiguration model with different parameter sources and modules.

In [ ]:
import sys
sys.path.append('..')
from pydantic import RootModel, Field
import typing as t


class ParameterSource(RootModel):
    root: "_TCache"


_TCacheUnion = None
_TCache = None
_default_root_field = ParameterSource.__pydantic_fields__["root"]


def register_cache_provider(source):
    global _TCacheUnion, _TCache
    if _TCacheUnion is None: _TCacheUnion = source
    # Note: it seems python is smart enough to optimize unions
    # so t.Union[A, t.Union[B, C]] is exactly t.Union[A, B, C] and doesn't create nested unions
    _TCacheUnion = t.Union[_TCacheUnion, source]
    _TCache = t.Annotated[_TCacheUnion, Field(discriminator="type")]
    # Have to reset this each time as pydantic "hard codes" the field after each time
    ParameterSource.__pydantic_fields__["root"] = _default_root_field
    print(_default_root_field)
    ParameterSource.model_rebuild(force=True)

In [4]:
from ped.param.sources.core import BaseSource
from ped.param.sources.request import RequestSource
from ped.param.sources.static import StaticSource

register_cache_provider(RequestSource)
register_cache_provider(StaticSource)

In [5]:
ParameterSource.model_validate({"type": "static"})
ParameterSource.model_validate({"type": "request"})

ValidationError: 1 validation error for ParameterSource
  Input should be None [type=none_required, input_value={'type': 'static'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/none_required

In [1]:
import sys
sys.path.append('..')

from ped.flow import FlowConfiguration
import json
from pprint import pprint

Registering RequestSource
True <class 'ped.param.sources.request.RequestSource'>
Registering StaticSource
True typing.Union[ped.param.sources.request.RequestSource, ped.param.sources.static.StaticSource]


In [2]:
from ped.param.sources import ParameterSource

In [9]:
from ped.param.sources import ParameterSource
from ped.param.sources.request import RequestSource
from ped.param.sources.static import StaticSource
import typing as t
from pydantic import Field

In [3]:
_annotated_type = t.Annotated[t.Union[RequestSource, StaticSource], Field(discriminator="type")]
# ExtendableModel.model_rebuild(force=True)

NameError: name 't' is not defined

In [15]:
ParameterSource.model_rebuild(force=True, _types_namespace={"_annotated_type": _annotated_type})

True

In [5]:
ParameterSource.model_validate(test_config["parameter_sources"]["static_config"])

ValidationError: 1 validation error for ParameterSource
  Input tag 'static' found using 'type' does not match any of the expected tags: 'request' [type=union_tag_invalid, input_value={'type': 'static', 'value...4.5, 'high_risk': 6.8}}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/union_tag_invalid

## Test Configuration Dictionary

Let's create a test configuration with:
- Static parameter source
- Request parameter source
- A few modules with different configurations

In [4]:
test_config = {
    "metadata": {
        "name": "Risk Assessment Flow",
        "description": "A flow for assessing customer risk using decision tables and inline calculations",
        "author": "Risk Team",
        "license": "MIT",
        "version": "1.0"
    },
    "parameter_sources": {
        "static_config": {
            "type": "static",
            "values": {
                "max_risk_score": 100,
                "default_threshold": 0.7,
                "min_credit_score": 650,
                "max_loan_amount": 500000,
                "interest_rates": {
                    "low_risk": 3.2,
                    "medium_risk": 4.5,
                    "high_risk": 6.8
                }
            }
        },
        "request_params": {
            "type": "request",
            "base_key": "parameters",
            "version_key": "version",
            "defaults": {
                "environment": "production",
                "debug_mode": False,
                "max_processing_time": 5000
            }
        }
    },
    "modules": {
        "risk_calculator": {
            "type": "inline",
            "module_config": {
                "expression_set": {
                    "credit_score_factor": "col('credit_score') / parameter('max_risk_score')",
                    "income_factor": "col('income') / 100000",
                    "risk_score": "(col('credit_score_factor') * 0.6) + (col('income_factor') * 0.4)"
                }
            },
            "input_mapping": {
                "credit_score": "customer_credit_score",
                "income": "customer_income"
            },
            "output_mapping": {
                "risk_score": "calculated_risk_score"
            },
            "parameters": {
                "max_risk_score": {
                    "source": "parameter_source",
                    "source_name": "static_config",
                    "key": "max_risk_score"
                },
                "threshold": {
                    "source": "parameter_source",
                    "source_name": "static_config", 
                    "key": "default_threshold"
                }
            }
        },
        "loan_decision": {
            "type": "decision_table",
            "module_config": {
                "parameters": {
                    "columns": ["age_min", "age_max", "income_min", "credit_min", "decision", "interest_rate", "max_loan"],
                    "values": [
                        [25, 35, 50000, 650, "approve", 3.5, 400000],
                        [30, 50, 70000, 700, "approve", 3.2, 600000],
                        [45, 65, 80000, 680, "approve", 3.8, 500000],
                        [18, 70, 100000, 600, "approve", 3.0, 800000],
                        [25, 45, 40000, 700, "conditional", 4.2, 250000]
                    ],
                    "dtypes": {
                        "age_min": "float",
                        "age_max": "float",
                        "income_min": "float",
                        "credit_min": "float",
                        "decision": "string",
                        "interest_rate": "float",
                        "max_loan": "float"
                    }
                },
                "expression": {
                    "typ": "and",
                    "expressions": [
                        {
                            "typ": "between",
                            "variable": "age",
                            "lower_bound_column": "age_min",
                            "upper_bound_column": "age_max",
                            "mode": "both_inclusive"
                        },
                        {
                            "typ": "between",
                            "variable": "income",
                            "lower_bound_column": "income_min",
                            "mode": "lower_inclusive"
                        },
                        {
                            "typ": "between",
                            "variable": "credit_score",
                            "lower_bound_column": "credit_min",
                            "mode": "lower_inclusive"
                        }
                    ]
                },
                "outputs": ["decision", "interest_rate", "max_loan"],
                "default": ["reject", 5.0, 0]
            },
            "input_mapping": {
                "age": "customer_age",
                "income": "customer_income",
                "credit_score": "customer_credit_score"
            },
            "output_mapping": {
                "decision": "loan_decision",
                "interest_rate": "approved_interest_rate",
                "max_loan": "approved_loan_amount"
            },
            "parameters": {
                "min_credit_score": {
                    "source": "parameter_source",
                    "source_name": "static_config",
                    "key": "min_credit_score"
                },
                "environment": {
                    "source": "parameter_source",
                    "source_name": "request_params",
                    "key": "environment"
                }
            }
        },
        "risk_adjuster": {
            "type": "my_custom_module",
            "version": "1.2",
            "module_config": {
                "adjustment_factors": {
                    "age_penalty": 0.1,
                    "employment_bonus": 0.05
                },
                "risk_categories": ["low", "medium", "high"]
            },
            "input_mapping": {
                "base_risk": "calculated_risk_score",
                "customer_age": "age"
            },
            "output_mapping": {
                "adjusted_risk": "final_risk_score"
            },
            "parameters": {
                "max_adjustment": {
                    "source": "value",
                    "value": 0.25
                },
                "interest_rates": {
                    "source": "parameter_source",
                    "source_name": "static_config",
                    "key": "interest_rates"
                }
            }
        }
    }
}

print("Test configuration created successfully!")
print(f"Number of parameter sources: {len(test_config['parameter_sources'])}")
print(f"Number of modules: {len(test_config['modules'])}")

Test configuration created successfully!
Number of parameter sources: 2
Number of modules: 3


## Validate Configuration with Pydantic

Now let's validate the configuration using FlowConfiguration.model_validate()

In [ ]:
from .param.sources import ParameterSource

In [4]:
try:
    flow_config = FlowConfiguration.model_validate(test_config)
    print("✅ Configuration validated successfully!")
    
    # Display the parsed configuration
    print("\n=== Parsed Flow Configuration ===")
    print(f"Name: {flow_config.metadata.name}")
    print(f"Description: {flow_config.metadata.description}")
    print(f"Author: {flow_config.metadata.author}")
    print(f"Version: {flow_config.metadata.version}")
    
    print("\n=== Parameter Sources ===")
    for name, source in flow_config.parameter_sources.items():
        print(f"{name}: {source.type}")
        if hasattr(source, 'values'):
            print(f"  - Static values: {list(source.values.keys())}")
        elif hasattr(source, 'base_key'):
            print(f"  - Request base_key: {source.base_key}")
    
    print("\n=== Modules ===")
    for name, module in flow_config.modules.items():
        print(f"{name}: {module.get('type', 'unknown')}")
        if 'parameters' in module:
            print(f"  - Parameters: {list(module['parameters'].keys())}")
            
except Exception as e:
    print(f"❌ Configuration validation failed: {e}")
    import traceback
    traceback.print_exc()

❌ Configuration validation failed: 1 validation error for FlowConfiguration
parameter_sources.static_config
  Input tag 'static' found using 'type' does not match any of the expected tags: 'request' [type=union_tag_invalid, input_value={'type': 'static', 'value...4.5, 'high_risk': 6.8}}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/union_tag_invalid


Traceback (most recent call last):
  File "/var/folders/mf/t5j3lfmd2wb_bzv520_hq19h0000gn/T/ipykernel_21488/1830809425.py", line 2, in <module>
    flow_config = FlowConfiguration.model_validate(test_config)
  File "/opt/miniconda/envs/dspdev/lib/python3.10/site-packages/pydantic/main.py", line 705, in model_validate
    return cls.__pydantic_validator__.validate_python(
pydantic_core._pydantic_core.ValidationError: 1 validation error for FlowConfiguration
parameter_sources.static_config
  Input tag 'static' found using 'type' does not match any of the expected tags: 'request' [type=union_tag_invalid, input_value={'type': 'static', 'value...4.5, 'high_risk': 6.8}}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/union_tag_invalid


## Test get_parameterized_modules()

Now let's test the parameterization functionality with sample request data.

In [ ]:
# Test with sample request data
sample_request = {
    "parameters": {
        "environment": "staging",
        "debug_mode": True,
        "max_processing_time": 3000,
        "version": "v1.2"
    },
    "customer_data": {
        "age": 32,
        "income": 75000,
        "credit_score": 720
    }
}

try:
    print("Testing get_parameterized_modules()...")
    parameterized_modules = flow_config.get_parameterized_modules(
        request=sample_request
    )
    
    print("✅ Parameterization successful!")
    print("\n=== Parameterized Modules ===")
    
    for module_name, module_config in parameterized_modules.items():
        print(f"\n--- {module_name} ---")
        print(f"Type: {module_config.get('type')}")
        
        if 'parameters' in module_config:
            print("Parameters:")
            for param_name, param_value in module_config['parameters'].items():
                print(f"  {param_name}: {param_value}")
        
        if 'input_mapping' in module_config:
            print(f"Input mapping: {module_config['input_mapping']}")
            
        if 'output_mapping' in module_config:
            print(f"Output mapping: {module_config['output_mapping']}")
            
except Exception as e:
    print(f"❌ Parameterization failed: {e}")
    import traceback
    traceback.print_exc()

## Test Different Request Scenarios

In [ ]:
# Test with minimal request
minimal_request = {}

print("Testing with minimal (empty) request...")
try:
    minimal_modules = flow_config.get_parameterized_modules(request=minimal_request)
    print("✅ Minimal request handled successfully")
    
    # Check if default values are used
    risk_calc_params = minimal_modules['risk_calculator']['parameters']
    print(f"Max risk score from static source: {risk_calc_params.get('max_risk_score')}")
    print(f"Threshold from static source: {risk_calc_params.get('threshold')}")
    
except Exception as e:
    print(f"❌ Minimal request failed: {e}")

In [ ]:
# Test with version-specific request
versioned_request = {
    "parameters": {
        "environment": "development",
        "debug_mode": True,
        "version": "v2.0"
    }
}

print("Testing with version-specific request...")
try:
    versioned_modules = flow_config.get_parameterized_modules(
        request=versioned_request,
        requested_versions={"request_params": "v2.0"}
    )
    print("✅ Versioned request handled successfully")
    
    # Show how parameters are resolved
    loan_decision_params = versioned_modules['loan_decision']['parameters']
    print(f"Environment from request: {loan_decision_params.get('environment')}")
    print(f"Min credit score from static: {loan_decision_params.get('min_credit_score')}")
    
except Exception as e:
    print(f"❌ Versioned request failed: {e}")

## Inspect Parameter Resolution Details

In [ ]:
# Let's examine how the parameter resolution works in detail
print("=== Parameter Resolution Analysis ===")

test_request = {
    "parameters": {
        "environment": "production",
        "debug_mode": False,
        "version": "v1.0"
    }
}

resolved_modules = flow_config.get_parameterized_modules(request=test_request)

print("\n--- Static Source Parameters ---")
static_source = flow_config.parameter_sources['static_config']
print(f"Type: {static_source.type}")
print(f"Available values: {list(static_source.values.keys())}")

print("\n--- Request Source Parameters ---")
request_source = flow_config.parameter_sources['request_params']
print(f"Type: {request_source.type}")
print(f"Base key: {request_source.base_key}")
print(f"Version key: {request_source.version_key}")
print(f"Defaults: {request_source.defaults}")

print("\n--- Module Parameter Resolution ---")
for module_name, module_config in resolved_modules.items():
    if 'parameters' in module_config:
        print(f"\n{module_name}:")
        for param_name, resolved_value in module_config['parameters'].items():
            print(f"  {param_name}: {resolved_value} (type: {type(resolved_value).__name__})")

## Test Edge Cases and Error Handling

In [ ]:
# Test configuration with missing parameter source reference
invalid_config = {
    "metadata": {
        "name": "Invalid Flow",
        "description": "This should fail validation",
        "author": "Test"
    },
    "parameter_sources": {
        "source1": {
            "type": "static",
            "values": {"key1": "value1"}
        }
    },
    "modules": {
        "module1": {
            "type": "inline",
            "module_config": {
                "expression_set": {"test": "parameter('missing_param')"}
            },
            "parameters": {
                "missing_param": {
                    "source": "parameter_source",
                    "source_name": "nonexistent_source",  # This should cause validation error
                    "key": "some_key"
                }
            }
        }
    }
}

print("Testing invalid configuration (should fail validation)...")
try:
    invalid_flow = FlowConfiguration.model_validate(invalid_config)
    print("❌ Expected validation to fail, but it passed!")
except Exception as e:
    print(f"✅ Validation correctly failed: {e}")

## Summary

This notebook demonstrates:

1. **Configuration Validation**: The FlowConfiguration model correctly validates complex nested configurations
2. **Parameter Sources**: Both static and request parameter sources work as expected
3. **Module Parameterization**: The `get_parameterized_modules()` method correctly resolves parameters from different sources
4. **Request Handling**: Different request scenarios (empty, versioned, etc.) are handled properly
5. **Error Handling**: Invalid configurations are caught and reported appropriately

The flow configuration system is working as designed!